In [0]:
### From Data Source (DB/Volume) to Bronze Table 
# - Format conversion from csv to delta
# - Adding metadata columns 

from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp, col

spark = SparkSession.builder.getOrCreate()

tables = ["customers","sales","sales_orders"]
source_path = "/Volumes/ecommerce_analytics/bronze/raw_data/"
catelog = "ecommerce_analytics"
schema = "bronze"
    
for table in tables:
    print(f"Processing {table}")
    input_path = f"{source_path}{table}"
    print(input_path)
    df = spark.read.csv(input_path, inferSchema=True, header=True)
    print(f"Read completed for {table}, Now adding metadata columns")

    df = df.withColumn("last_update_ts", current_timestamp())\
            .withColumn("file_path", col("_metadata.file_path"))
    print(f"Added metadata columns successfully, Now writing {table}")

    df.write.format("delta")\
        .mode("overwrite") \
        .saveAsTable(f"{catelog}.{schema}.{table}")
    print(f"Write completed for {table}")

print(f"Write completed for all {tables}")

In [0]:
### From Data Source (DB/Volume) to Bronze Table 
# - Format conversion from csv to delta
# - Adding metadata columns 
# - Following Good practice using functions

from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp, col

spark = SparkSession.builder.getOrCreate()

tables = ["customers", "sales", "sales_orders"]

source_path = "/Volumes/ecommerce_analytics/bronze/raw_data/"
catalog = "ecommerce_analytics"
schema = "bronze"

# Utility/Helper Functions

def add_metadata_columns(df):
    df = df.withColumn("last_update_ts",  current_timestamp())\
            .withColumn("file_path", col("_metadata.file_path"))    
    return df

for table in tables:
    print(f"Processing table {table}")
    input_path = f"{source_path}{table}"
    print(input_path)
    df = spark.read.csv(input_path, inferSchema=True, header=True)
    print(f"Read completed for {table}, Now adding metadata columns")

    df = add_metadata_columns(df)
    print(f"Added metadata columns successfully, Now writing {table}")

    df.write.format("delta")\
        .mode("overwrite")\
        .saveAsTable(f"{catalog}.{schema}.{table}")
    print(f"Write completed for {table}")

print(f"Write completed for all tables {tables}")
     

In [0]:
# reading the 3 created tables

df_customers = spark.read.table("ecommerce_analytics.bronze.customers")
display(df_customers)
df_sales = spark.read.table("ecommerce_analytics.bronze.sales")
display(df_sales)
df_sales_orders = spark.read.table("ecommerce_analytics.bronze.sales_orders")
display(df_sales_orders)